# 07 — Independent endothelial and astrocytic stroke validation: source audit

Two additional independent GEO experiments specifically addressing problems in the current screen:

- **GSE223744** — **8 samples, 4 ischemia and 4 control**, RNA-seq of FACS-purified mouse CD31+ brain endothelial cells, 24h after transient MCAO, linked to Arbaizar-Rovirosa et al., Journal of Neuroinflammation 2023. This directly tests vascular target gene biology, avoiding reliance on broad cell-cluster annotation.
- **GSE286075** — **6 samples from 3 animals, paired ipsilateral vs contralateral** astrocytic endfoot RiboTag translatome, 2h ischemia +6h reperfusion, Shim et al. 2025. This tests early astrocyte target biology rather than pretending global astrocyte scRNA equals endfeet.

**THIS NOTEBOOK IS A PROVENANCE AND FORMAT AUDIT.** The ensuing analytical notebook will use exact sample phenotype and actual raw/processed count file schema determined here.

No samples merged across datasets; no biological inference from raw file inventory. Source details from GEO SOFT and exact SHA256.

GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE223744
GEO: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE286075


In [ ]:
import requests,gzip,io,tarfile,hashlib,json,os
from pathlib import Path
import pandas as pd
ROOT=Path(os.environ.get("IMACS_ROOT","/content/imacs_stroke_pet_validation" if Path("/content").exists() else "imacs_stroke_pet_validation"))
RAW=ROOT/"raw"; OUT=ROOT/"results"
RAW.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
SOURCES=["GSE223744","GSE286075"]
def url(acc,folder,ext):
    return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{acc[:-3]}nnn/{acc}/{folder}/{acc}_{ext}"
def download(u,p):
    if p.exists() and p.stat().st_size>100:return p
    with requests.get(u,timeout=240,stream=True) as r:
        r.raise_for_status()
        with p.open("wb") as f:
            for b in r.iter_content(1<<20):
                if b:f.write(b)
    return p
def sha(p):
    h=hashlib.sha256()
    with p.open("rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):h.update(b)
    return h.hexdigest()


In [ ]:
sample_info=[]; inventory=[]; provenance=[]; headers=[]
for accession in SOURCES:
    soft=download(url(accession,"soft","family.soft.gz"),RAW/f"{accession}_family.soft.gz")
    archive=download(url(accession,"suppl","RAW.tar"),RAW/f"{accession}_RAW.tar")
    for p in [soft,archive]:
        provenance.append({"accession":accession,"file":p.name,"bytes":p.stat().st_size,"sha256":sha(p)})
    current=None
    with gzip.open(soft,"rt",errors="replace") as f:
        for line in f:
            line=line.rstrip("\n")
            if line.startswith("^SAMPLE = "):
                if current:sample_info.append(current)
                current={"accession":accession,"gsm":line.split("=",1)[1].strip(),"characteristics":[],"supplementary":[]}
            elif current and line.startswith("!Sample_"):
                key,_,value=line[1:].partition(" = ")
                if key=="Sample_characteristics_ch1":current["characteristics"].append(value)
                elif key.startswith("Sample_supplementary_file"):current["supplementary"].append(value)
                elif key in ("Sample_title","Sample_source_name_ch1","Sample_description","Sample_treatment_protocol"):
                    current[key.replace("Sample_","")]=value
        if current:sample_info.append(current)
    with tarfile.open(archive,"r:*") as tar:
        for m in tar:
            if not m.isfile():continue
            inventory.append({"accession":accession,"name":m.name,"bytes":m.size})
            handle=tar.extractfile(m)
            if handle is None:continue
            preview=handle.read(min(4096,m.size))
            if m.name.endswith(".gz"):
                try:preview=gzip.decompress(preview)
                except EOFError:preview=b"__COMPRESSED_PREVIEW_NEEDS_STREAM__"
            headers.append({"accession":accession,"name":m.name,"first_lines":preview.decode("utf-8",errors="replace")[:600]})
    print(accession,"samples",sum(x["accession"]==accession for x in sample_info),
          "tar files",sum(x["accession"]==accession for x in inventory))

m=pd.DataFrame(sample_info)
m["characteristics"]=m.characteristics.apply(lambda v:" | ".join(v))
m["supplementary"]=m.supplementary.apply(lambda v:" | ".join(v))
m.to_csv(OUT/"07_GEO_sample_metadata.csv",index=False)
pd.DataFrame(inventory).to_csv(OUT/"07_GEO_RAW_tar_inventory.csv",index=False)
pd.DataFrame(headers).to_csv(OUT/"07_GEO_RAW_preview.csv",index=False)
pd.DataFrame(provenance).to_csv(OUT/"07_GEO_RAW_checksums.csv",index=False)
display(m[["accession","gsm","title","characteristics","supplementary"]])
display(pd.DataFrame(inventory))
display(pd.DataFrame(headers).head(8))

### Gate

Do not interpret `G1`/`G3` or ipsilateral/contralateral until read from **sample metadata**. No raw RNA table should be treated as counts if values are already normalized. The full analysis must respect four independent endothelial animals/group and three **paired** astrocyte samples per group (if titles/metadata confirm pairs).